In [1]:
import pandas as pd
import os

def get_top_k_summary_df(results_save_dp):
    results_summary_fp = os.path.join(results_save_dp,"overall_stats_summary.csv")
    results_summary_df = pd.read_csv(results_summary_fp)
    results_summary_df['seed'] = results_summary_df['model'].apply(lambda x: x.split('_')[-1])
    results_summary_df['model'] = results_summary_df['model'].apply(lambda x: x.rsplit('_', 1)[0])
    #results_summary_df = results_summary_df[results_summary_df['ranking method'] == 'cos_sim_0.01']
    return results_summary_df

def get_mean_median_summary_df(results_save_dp):
    fnames = [f for f in os.listdir(results_save_dp) if f != "overall_stats_summary.csv"]
    stats_data = []
    for fname in fnames:
        fname_no_ext = fname.split('.')[0]
        model, seed, _ = fname_no_ext.rsplit("_",2)
        
        df = pd.read_csv( os.path.join(results_save_dp,fname))
        #cos_hun_rank 
        cos_bin_rank_mean = df['cos_sim_0.01_rank'].mean()
        cos_bin_rank_median = df['cos_sim_0.01_rank'].median()
        cos_hun_rank_mean = df['cos_hun_rank'].mean()
        cos_hun_rank_median = df['cos_hun_rank'].median()
        stats_data.append((model, seed, cos_bin_rank_mean,cos_bin_rank_median,cos_hun_rank_mean,cos_hun_rank_median))
        ##print(df)
        #break
        #print(fps)
    #results_summary_fp = os.path.join(results_save_dp,"overall_stats_summary.csv")
    #results_summary_df = pd.read_csv(results_summary_fp)
    #results_summary_df['seed'] = results_summary_df['model'].apply(lambda x: x.split('_')[-1])
    #results_summary_df['model'] = results_summary_df['model'].apply(lambda x: x.rsplit('_', 1)[0])
    ##results_summary_df = results_summary_df[results_summary_df['ranking method'] == 'cos_sim_0.01']
    #return results_summary_df
    stats_data_df = pd.DataFrame(data=stats_data,columns=['model', 'seed', 'cos_bin_rank_mean','cos_bin_rank_median','cos_hun_rank_mean','cos_hun_rank_median'])
    stats_data_df = stats_data_df.sort_values(by=['model','seed'])
    return stats_data_df

## Inchikey Split

In [2]:
results_save_dp = "../data/ms2c/pubchem/run_240925/benchmark/pubchem_nist20mona23v3_d4_wr00_inchikey_test_10ppm_MORGAN-R2_50"
top_k_summary_df = get_top_k_summary_df(results_save_dp)
#pd.options.display.float_format = '{:.2%}'.format
top_k_grouped = top_k_summary_df.groupby(['model','ranking method']).agg({'top-1 ratio':['mean', 'std'],
                                                              'top-3 ratio':['mean', 'std'],
                                                              'top-5 ratio':['mean', 'std'],
                                                              'top-10 ratio':['mean', 'std'],
                                                              'top-10 ratio':['mean', 'std'],
                                                              'ans_score_mean':['mean'],
                                                              'ndcg_linear_mean':['mean'],
                                                              'ndcg_exp_mean':['mean']})
top_k_grouped

top-1 ratio           top-3 ratio            \
                                  mean       std        mean       std   
model       ranking method                                               
fragnnet_d3 cos_hun           0.239421  0.003315    0.518501  0.004460   
            cos_sim_0.01      0.235009  0.003759    0.513899  0.003607   
fragnnet_d4 cos_hun           0.239753  0.004281    0.514421  0.003377   
            cos_sim_0.01      0.235057  0.004213    0.511717  0.002266   
iceberg     cos_hun           0.245114  0.008578    0.509440  0.003076   
            cos_sim_0.01      0.249194  0.008410    0.515560  0.002888   
massformer  cos_hun           0.084345  0.002692    0.223861  0.002883   
            cos_sim_0.01      0.154839  0.004882    0.366034  0.009728   
neims       cos_hun           0.095636  0.002897    0.235816  0.002444   
            cos_sim_0.01      0.177040  0.003220    0.383871  0.005083   

                           top-5 ratio           top-10 ratio            \
                                  mean       std         mean       std   
model       ranking method                                                
fragnnet_d3 cos_hun           0.659677  0.003488     0.830977  0.004906   
            cos_sim_0.01      0.656262  0.003224     0.828416  0.004778   
fragnnet_d4 cos_hun           0.658491  0.007543     0.837666  0.002352   
            cos_sim_0.01      0.660057  0.004585     0.831072  0.001130   
iceberg     cos_hun           0.643264  0.004535     0.808918  0.004235   
            cos_sim_0.01      0.643074  0.006473     0.806973  0.004730   
massformer  cos_hun           0.329412  0.003804     0.526518  0.005321   
            cos_sim_0.01      0.495778  0.008515     0.689991  0.006671   
neims       cos_hun           0.335531  0.004834     0.511765  0.004124   
            cos_sim_0.01      0.502799  0.003974     0.675854  0.004799   

                           ans_score_mean ndcg_linear_mean ndcg_exp_mean  
                                     mean             mean          mean  
model       ranking method                                                
fragnnet_d3 cos_hun              0.683041         0.916676      0.896300  
            cos_sim_0.01         0.711314         0.915949      0.895410  
fragnnet_d4 cos_hun              0.697223         0.917096      0.896821  
            cos_sim_0.01         0.725311         0.916191      0.895715  
iceberg     cos_hun              0.652593         0.914361      0.893598  
            cos_sim_0.01         0.688906         0.914566      0.893890  
massformer  cos_hun              0.319068         0.875805      0.846105  
            cos_sim_0.01         0.628855         0.896306      0.871115  
neims       cos_hun              0.315601         0.877004      0.847533  
            cos_sim_0.01         0.626647         0.898439      0.873748

* inchikey linear ncdg @50
    * worest: 0.7876979546374592
    * random: 0.8420145176382821
* inchikey exp ncdg @50
    * worest: 0.7451008867236989
    * random: 0.8051250529492999

In [3]:
top_k_grouped['ndcg_linear_mean']['mean'] - 0.7876979546374592

model        ranking method
fragnnet_d3  cos_hun           0.128978
             cos_sim_0.01      0.128251
fragnnet_d4  cos_hun           0.129398
             cos_sim_0.01      0.128493
iceberg      cos_hun           0.126663
             cos_sim_0.01      0.126868
massformer   cos_hun           0.088107
             cos_sim_0.01      0.108608
neims        cos_hun           0.089306
             cos_sim_0.01      0.110741
Name: mean, dtype: float64

In [4]:
top_k_grouped['ndcg_exp_mean']['mean'] - 0.7792600125793003

model        ranking method
fragnnet_d3  cos_hun           0.117040
             cos_sim_0.01      0.116150
fragnnet_d4  cos_hun           0.117561
             cos_sim_0.01      0.116455
iceberg      cos_hun           0.114338
             cos_sim_0.01      0.114630
massformer   cos_hun           0.066845
             cos_sim_0.01      0.091855
neims        cos_hun           0.068273
             cos_sim_0.01      0.094488
Name: mean, dtype: float64

### Top-K

In [5]:
top_k_summary_df[(top_k_summary_df['seed'] == 's0') & (top_k_summary_df['ranking method'] == 'cos_sim_0.01')]

,model,ranking method,top-1,top-3,top-5,top-10,top-1 ratio,top-3 ratio,top-5 ratio,top-10 ratio,ans_score_mean,ans_score_std,top_1_tanimoto_mean,top_1_tanimoto_std,ndcg_linear_mean,ndcg_linear_std,ndcg_exp_mean,ndcg_exp_std,seed
1,fragnnet_d3,cos_sim_0.01,983,2171,2749,3490,0.233159,0.514943,0.652040,0.827799,0.710142,0.208738,0.579883,0.272628,0.915531,0.055283,0.894900,0.067751,s0
11,fragnnet_d4,cos_sim_0.01,1003,2164,2762,3496,0.237903,0.513283,0.655123,0.829222,0.722858,0.205011,0.582729,0.273752,0.916011,0.054773,0.895551,0.067065,s0
21,iceberg,cos_sim_0.01,1074,2184,2730,3403,0.254744,0.518027,0.647533,0.807163,0.690503,0.217673,0.594359,0.277360,0.914983,0.054102,0.894402,0.066872,s0
31,massformer,cos_sim_0.01,688,1612,2146,2948,0.163188,0.382353,0.509013,0.699241,0.632908,0.248853,0.515323,0.260324,0.897508,0.061462,0.872682,0.074098,s0
41,neims,cos_sim_0.01,728,1616,2095,2846,0.172676,0.383302,0.496917,0.675047,0.627422,0.246291,0.528826,0.262801,0.898221,0.061689,0.873465,0.074577,s0


In [6]:
top_k_summary_df[(top_k_summary_df['seed'] == 's0') & (top_k_summary_df['ranking method'] == 'cos_hun')]

,model,ranking method,top-1,top-3,top-5,top-10,top-1 ratio,top-3 ratio,top-5 ratio,top-10 ratio,ans_score_mean,ans_score_std,top_1_tanimoto_mean,top_1_tanimoto_std,ndcg_linear_mean,ndcg_linear_std,ndcg_exp_mean,ndcg_exp_std,seed
0,fragnnet_d3,cos_hun,1022,2199,2771,3477,0.242410,0.521584,0.657258,0.824715,0.682808,0.199327,0.585195,0.274763,0.916586,0.054207,0.896219,0.066785,s0
10,fragnnet_d4,cos_hun,1012,2165,2751,3544,0.240038,0.513520,0.652514,0.840607,0.695321,0.193988,0.586481,0.273503,0.916915,0.053282,0.896628,0.065581,s0
20,iceberg,cos_hun,1050,2146,2717,3422,0.249051,0.509013,0.644450,0.811670,0.653714,0.208329,0.590923,0.275921,0.914698,0.053235,0.894016,0.065865,s0
30,massformer,cos_hun,357,955,1415,2256,0.084677,0.226518,0.335626,0.535104,0.322297,0.242031,0.437490,0.222160,0.876160,0.063404,0.846565,0.074812,s0
40,neims,cos_hun,384,984,1410,2170,0.091082,0.233397,0.334440,0.514706,0.316487,0.239120,0.448973,0.226781,0.876565,0.063677,0.847009,0.075559,s0


### Mean and Median

In [7]:
mean_median_summary_df = get_mean_median_summary_df(results_save_dp)
mean_median_summary_df

,model,seed,cos_bin_rank_mean,cos_bin_rank_median,cos_hun_rank_mean,cos_hun_rank_median
11,fragnnet_d3,s0,6.283918,3.0,6.152514,3.00
1,fragnnet_d3,s1,6.307875,3.0,6.185721,3.00
23,fragnnet_d3,s2,6.162120,3.0,6.080289,3.00
3,fragnnet_d3,s3,6.185009,3.0,6.124051,3.00
0,fragnnet_d3,s4,6.202324,3.0,6.089658,3.00
10,fragnnet_d4,s0,6.166983,3.0,5.995256,3.00
14,fragnnet_d4,s1,6.201613,3.0,6.049929,3.00
16,fragnnet_d4,s2,6.093098,3.0,5.973197,3.00
17,fragnnet_d4,s3,6.216793,3.0,6.101281,3.00
21,fragnnet_d4,s4,6.110769,3.0,6.022296,3.00


In [8]:
mean_median_grouped = mean_median_summary_df.groupby(['model']).agg({'cos_bin_rank_mean':['mean', 'std'],
                                                              'cos_bin_rank_median':['mean', 'std'],
                                                              'cos_hun_rank_mean':['mean', 'std'],
                                                              'cos_hun_rank_median':['mean', 'std']})
mean_median_grouped

cos_bin_rank_mean           cos_bin_rank_median            \
                         mean       std                mean       std   
model                                                                   
fragnnet_d3          6.228250  0.063941                 3.0  0.000000   
fragnnet_d4          6.157851  0.054503                 3.0  0.000000   
iceberg              6.638899  0.051104                 3.0  0.000000   
massformer           9.430977  0.129541                 5.8  0.447214   
neims                9.915299  0.126163                 5.2  0.447214   

            cos_hun_rank_mean           cos_hun_rank_median           
                         mean       std                mean      std  
model                                                                 
fragnnet_d3          6.126447  0.043825                3.00  0.00000  
fragnnet_d4          6.028392  0.049895                3.00  0.00000  
iceberg              6.618501  0.049749                3.00  0.00000  
massformer          13.671845  0.125288                9.85  0.33541  
neims               14.395351  0.076742               10.00  0.00000

## Scaffold Split

In [9]:
results_save_dp = "../data/ms2c/pubchem/run_240925/benchmark/pubchem_nist20mona23v3_d4_wr00_scaffold_test_10ppm_MORGAN-R2_50"
top_k_summary_df = get_top_k_summary_df(results_save_dp)
#pd.options.display.float_format = '{:.2%}'.format
top_k_grouped = top_k_summary_df.groupby(['model','ranking method']).agg({'top-1 ratio':['mean', 'std'],
                                                              'top-3 ratio':['mean', 'std'],
                                                              'top-5 ratio':['mean', 'std'],
                                                              'top-10 ratio':['mean', 'std'],
                                                              'top-10 ratio':['mean', 'std'],
                                                              'ans_score_mean':['mean'],
                                                              'ndcg_linear_mean':['mean'],
                                                              'ndcg_exp_mean':['mean']})
top_k_grouped

top-1 ratio           top-3 ratio            \
                                  mean       std        mean       std   
model       ranking method                                               
fragnnet_d3 cos_hun           0.204370  0.003177    0.458665  0.005340   
            cos_sim_0.01      0.199940  0.003885    0.457588  0.006021   
fragnnet_d4 cos_hun           0.207543  0.005427    0.469380  0.009251   
            cos_sim_0.01      0.208022  0.004603    0.461000  0.010867   
iceberg     cos_hun           0.193655  0.002459    0.436815  0.007272   
            cos_sim_0.01      0.198324  0.002154    0.432445  0.005581   
massformer  cos_hun           0.069081  0.003711    0.196348  0.004690   
            cos_sim_0.01      0.117749  0.005423    0.297935  0.010882   
neims       cos_hun           0.070937  0.001881    0.182640  0.002755   
            cos_sim_0.01      0.125591  0.004745    0.289973  0.005565   

                           top-5 ratio           top-10 ratio            \
                                  mean       std         mean       std   
model       ranking method                                                
fragnnet_d3 cos_hun           0.605507  0.003510     0.794912  0.002561   
            cos_sim_0.01      0.602035  0.004508     0.791559  0.003064   
fragnnet_d4 cos_hun           0.617839  0.007965     0.799820  0.006757   
            cos_sim_0.01      0.608261  0.011915     0.789644  0.006291   
iceberg     cos_hun           0.574020  0.010289     0.756121  0.011266   
            cos_sim_0.01      0.566537  0.005581     0.751931  0.010350   
massformer  cos_hun           0.301467  0.008212     0.491769  0.008262   
            cos_sim_0.01      0.431188  0.006748     0.637594  0.007400   
neims       cos_hun           0.271835  0.006691     0.440946  0.005685   
            cos_sim_0.01      0.402215  0.006731     0.585633  0.003772   

                           ans_score_mean ndcg_linear_mean ndcg_exp_mean  
                                     mean             mean          mean  
model       ranking method                                                
fragnnet_d3 cos_hun              0.626212         0.906478      0.883978  
            cos_sim_0.01         0.652187         0.905986      0.883359  
fragnnet_d4 cos_hun              0.646680         0.907651      0.885409  
            cos_sim_0.01         0.671934         0.906898      0.884505  
iceberg     cos_hun              0.589538         0.903099      0.879824  
            cos_sim_0.01         0.621260         0.902854      0.879564  
massformer  cos_hun              0.281487         0.865664      0.834249  
            cos_sim_0.01         0.555540         0.883854      0.856109  
neims       cos_hun              0.269917         0.862772      0.830672  
            cos_sim_0.01         0.541494         0.880834      0.852516

* scaffold linear ncdg @50
    * worest: 0.7792600125793003
    * random: 0.8353117047278623
* scaffold exp ncdg @50
    * worest: 0.7357161420498771
    * random: 0.7982340648434951

In [10]:
top_k_grouped['ndcg_linear_mean']['mean'] - 0.7792600125793003

model        ranking method
fragnnet_d3  cos_hun           0.127218
             cos_sim_0.01      0.126726
fragnnet_d4  cos_hun           0.128391
             cos_sim_0.01      0.127638
iceberg      cos_hun           0.123839
             cos_sim_0.01      0.123594
massformer   cos_hun           0.086404
             cos_sim_0.01      0.104594
neims        cos_hun           0.083512
             cos_sim_0.01      0.101574
Name: mean, dtype: float64

In [11]:
top_k_grouped['ndcg_exp_mean']['mean'] - 0.7357161420498771

model        ranking method
fragnnet_d3  cos_hun           0.148262
             cos_sim_0.01      0.147643
fragnnet_d4  cos_hun           0.149693
             cos_sim_0.01      0.148789
iceberg      cos_hun           0.144108
             cos_sim_0.01      0.143848
massformer   cos_hun           0.098533
             cos_sim_0.01      0.120393
neims        cos_hun           0.094956
             cos_sim_0.01      0.116800
Name: mean, dtype: float64

In [12]:
top_k_summary_df[(top_k_summary_df['seed'] == 's0') & (top_k_summary_df['ranking method'] == 'cos_sim_0.01')]

,model,ranking method,top-1,top-3,top-5,top-10,top-1 ratio,top-3 ratio,top-5 ratio,top-10 ratio,ans_score_mean,ans_score_std,top_1_tanimoto_mean,top_1_tanimoto_std,ndcg_linear_mean,ndcg_linear_std,ndcg_exp_mean,ndcg_exp_std,seed
1,fragnnet_d3,cos_sim_0.01,686,1534,2003,2646,0.205328,0.459144,0.599521,0.791978,0.653994,0.216397,0.547963,0.272605,0.906833,0.057838,0.884393,0.070622,s0
11,fragnnet_d4,cos_sim_0.01,712,1565,2077,2663,0.213110,0.468423,0.621670,0.797067,0.671859,0.212660,0.554036,0.274202,0.907308,0.059066,0.885041,0.071896,s0
21,iceberg,cos_sim_0.01,658,1434,1892,2472,0.196947,0.429213,0.566298,0.739898,0.619977,0.228375,0.546280,0.268138,0.902511,0.058257,0.879134,0.071005,s0
31,massformer,cos_sim_0.01,415,1022,1480,2160,0.124214,0.305896,0.442981,0.646513,0.559419,0.238847,0.468084,0.246043,0.885079,0.062168,0.857602,0.074264,s0
41,neims,cos_sim_0.01,440,965,1337,1957,0.131697,0.288836,0.400180,0.585753,0.544446,0.241259,0.475006,0.250018,0.881695,0.064005,0.853536,0.076688,s0


In [13]:
top_k_summary_df[(top_k_summary_df['seed'] == 's0') & (top_k_summary_df['ranking method'] == 'cos_hun')]

,model,ranking method,top-1,top-3,top-5,top-10,top-1 ratio,top-3 ratio,top-5 ratio,top-10 ratio,ans_score_mean,ans_score_std,top_1_tanimoto_mean,top_1_tanimoto_std,ndcg_linear_mean,ndcg_linear_std,ndcg_exp_mean,ndcg_exp_std,seed
0,fragnnet_d3,cos_hun,696,1556,2025,2651,0.208321,0.465729,0.606106,0.793475,0.627293,0.207354,0.547525,0.274209,0.906839,0.057564,0.884440,0.070330,s0
10,fragnnet_d4,cos_hun,684,1596,2090,2698,0.204729,0.477701,0.625561,0.807543,0.645570,0.200928,0.548151,0.272618,0.907413,0.058427,0.885119,0.071126,s0
20,iceberg,cos_hun,638,1446,1896,2481,0.190961,0.432805,0.567495,0.742592,0.589481,0.216661,0.540082,0.266982,0.902711,0.057309,0.879336,0.069837,s0
30,massformer,cos_hun,215,637,964,1664,0.064352,0.190661,0.288536,0.498054,0.283419,0.225752,0.409198,0.208649,0.865935,0.063836,0.834456,0.074738,s0
40,neims,cos_hun,231,605,907,1465,0.069141,0.181084,0.271476,0.438491,0.271674,0.222070,0.412797,0.211940,0.863086,0.064373,0.830969,0.075652,s0


In [14]:
mean_median_summary_df = get_mean_median_summary_df(results_save_dp)
mean_median_summary_df

,model,seed,cos_bin_rank_mean,cos_bin_rank_median,cos_hun_rank_mean,cos_hun_rank_median
11,fragnnet_d3,s0,6.932356,4.0,6.870398,4.0
1,fragnnet_d3,s1,7.020652,4.0,6.953906,4.0
23,fragnnet_d3,s2,7.012870,4.0,6.921281,4.0
3,fragnnet_d3,s3,6.948219,4.0,6.868303,4.0
0,fragnnet_d3,s4,7.101766,4.0,7.017959,4.0
10,fragnnet_d4,s0,6.855881,4.0,6.644717,4.0
14,fragnnet_d4,s1,7.081413,4.0,6.859024,4.0
16,fragnnet_d4,s2,7.312182,4.0,7.009578,4.0
17,fragnnet_d4,s3,6.835229,4.0,6.599222,4.0
21,fragnnet_d4,s4,7.135588,4.0,6.972763,4.0


In [15]:
mean_median_grouped = mean_median_summary_df.groupby(['model']).agg({'cos_bin_rank_mean':['mean', 'std'],
                                                              'cos_bin_rank_median':['mean', 'std'],
                                                              'cos_hun_rank_mean':['mean', 'std'],
                                                              'cos_hun_rank_median':['mean', 'std']})
mean_median_grouped

cos_bin_rank_mean           cos_bin_rank_median       \
                         mean       std                mean  std   
model                                                              
fragnnet_d3          7.003173  0.067371                 4.0  0.0   
fragnnet_d4          7.044059  0.200427                 4.0  0.0   
iceberg              7.867884  0.203494                 4.0  0.0   
massformer          10.559773  0.173572                 7.0  0.0   
neims               11.954714  0.061587                 8.0  0.0   

            cos_hun_rank_mean           cos_hun_rank_median            
                         mean       std                mean       std  
model                                                                  
fragnnet_d3          6.926369  0.062604                 4.0  0.000000  
fragnnet_d4          6.817061  0.187232                 4.0  0.000000  
iceberg              7.707872  0.149621                 4.0  0.000000  
massformer          14.237294  0.163938                10.8  0.447214  
neims               16.008830  0.098692                12.8  0.447214